# Benchmark shortcuts and transliteration consistency in zero-shot cross-script classification of Moroccan Darija

Soufiane Hourri  
Abdelhay Zoizou  
Ilham El Ouariachi  
Brahim Ait Skourt  
Moad Hicham Safhi  
Mohammed Bergui

Contact: Soufiane Hourri (s.hourri@uca.ac.ma)

This notebook reproduces every experiment, table and figure of the paper. It runs on Google Colab Pro (an A100 is recommended; an L4 also works but is slower) and is fully automatic. Run all cells. The datasets download themselves, and no manual annotation, hand-written rules or hand-tuned weights are needed.

### What the notebook does

Moroccan Darija is written online in Arabic script and in a Latin-script form called Arabizi, which has no standard orthography. A classifier trained on labelled text in one script is evaluated on naturally written text in the other script, with model selection on the source script only.

The pipeline has four parts.

1. An automatic, model-free audit of each dataset and direction looks for cues that are identical in both scripts, such as emojis, Latin tokens inside Arabic-script text, mixed-script items and near-duplicates. A direction whose character n-gram cross-script AUROC reaches the contamination threshold is flagged, and a cleaned copy of the task is added when stripping the other script's letters removes the cue.
2. A stochastic Arabic to Arabizi transliterator is learned from the public DoDa parallel corpus by hard-EM Viterbi alignment. It is evaluated against uroman, against a ByT5-small model trained on the same pairs, and on a multi-reference lexicon of human spellings. The sampling temperature is chosen on DoDa development pairs without task data.
3. Encoders are fine-tuned on each labelled example together with a transliterated view, in a full 2×2×2 factorial design over transliteration (argmax or sampled), a symmetric KL consistency term and an InfoNCE alignment term. The configuration with sampling and both terms is called ScriptBridge. Baselines include vanilla fine-tuning, test-time transliteration, a TransliCo-style system with uroman, R-Drop, character noise and a supervised dual-script reference, which is a diagnostic comparator and not a zero-shot system.
4. Six primary contrasts declared in advance are tested with a hierarchical bootstrap over seeds and test examples and Holm correction. Script invariance is measured on 1,000 human-written DoDa sentence pairs.

### Data, downloaded automatically

| Dataset | Use | Source |
|---|---|---|
| MYC, Moroccan YouTube Corpus (Jbel et al.) | sentiment task, both scripts | `github.com/MouadJb/MYC` |
| Moroccan Darija Offensive Language (Ibrahimi and Mourhir), version 3 with the original human labels | offensive-language task, both scripts | DOI 10.17632/2y4m97b7dc.3 |
| DoDa, Darija Open Dataset | transliterator training and evaluation, temperature selection, invariance measurement; never used as labelled task examples | `github.com/darija-open-dataset/dataset` |

The texts are not redistributed with this repository.

In [ ]:
!pip -q install -U transformers accelerate uroman sentencepiece protobuf   # do NOT upgrade pandas/matplotlib (breaks Colab)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Configuration
The defaults are the paper settings. Set `QUICK_RUN=True` for a ~10-minute end-to-end check on a subsample.

In [ ]:
import os, json, math, random, re, time, copy, io, pickle
from dataclasses import dataclass, asdict
import numpy as np, pandas as pd

@dataclass
class CFG:
    QUICK_RUN: bool = False
    TASKS: tuple = ("myc", "offensive")      # "offensive" is skipped automatically if download fails
    MYC_URL: str = "https://raw.githubusercontent.com/MouadJb/MYC/main/DATA_CLEANED.csv"
    DODA_URL: str = "https://raw.githubusercontent.com/darija-open-dataset/dataset/main/sentences/sentences.csv"
    OFFENSIVE_MENDELEY: str = "https://data.mendeley.com/public-api/datasets/2y4m97b7dc/files?folder_id=root&version=3"
    TEST_FRAC: float = 0.2
    DEV_FRAC: float = 0.1
    MIN_CHARS: int = 3
    MIN_TOKENS: int = 2
    STRIP_EMOJI: bool = True                 # MYC labels are ~fully predictable from emojis alone (see §5.1)
    # model
    MODELS: tuple = ("xlm-roberta-base", "SI2M-Lab/DarijaBERT", "SI2M-Lab/DarijaBERT-mix", "microsoft/mdeberta-v3-base")
    MAX_MODELS: int = 3                      # run the first 3 that pass the tokenizer-coverage check
    FULL_GRID_MODELS: int = 2                # the first 2 backbones get the full grid; later ones the reduced grid
    UNK_MAX: float = 0.05                    # max [UNK] rate allowed on either script
    MODEL_NAME: str = "xlm-roberta-base"     # set automatically in the run loop
    MAX_LEN: int = 128
    PROJ_DIM: int = 128
    # training
    EPOCHS: int = 4
    BATCH_SIZE: int = 32
    LR: float = 3e-5
    WEIGHT_DECAY: float = 0.01
    WARMUP: float = 0.1
    PATIENCE: int = 2
    GRAD_CLIP: float = 1.0
    # ScriptBridge (defaults; overwritten by AUTO_TUNE using source-dev only)
    LAMBDA_KL: float = 1.0
    LAMBDA_CON: float = 0.5
    TAU: float = 0.07
    AUTO_TUNE: bool = True
    TUNE_GRID: tuple = ((0.5, 0.1), (1.0, 0.5), (2.0, 1.0))
    # transliterator
    TR_EM_ITERS: int = 3
    TR_MAX_WORDS: int = 120000
    TR_TEST_PAIRS: int = 2000
    TR_DEV_PAIRS: int = 2000                 # DoDa dev pairs used ONLY to pick the sampling temperature
    AUTO_TEMP: bool = True
    TEMP_GRID: tuple = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5)   # 0.0 (argmax) is a legitimate candidate
    LEXICON_K: int = 10                      # samples per word for multi-reference lexicon evaluation
    RUN_BYT5: bool = True                    # same-data neural transliteration baseline (ByT5-small)
    BYT5_NAME: str = "google/byt5-small"
    BYT5_EPOCHS: int = 1
    BYT5_BATCH: int = 32
    BYT5_LR: float = 3e-4
    BYT5_MAXLEN: int = 256
    CHAR_NOISE_P: float = 0.10               # generic character-perturbation control
    CONTAM_AUROC: float = 0.75               # char n-gram cross-script AUROC above this => direction flagged as contaminated
    RUN_CLEANED: bool = True                 # if a flagged direction is fixed by stripping other-script chars, also run the cleaned task
    N_BOOT: int = 1000
    # grid
    DIRECTIONS: tuple = ("arabic", "latin")
    # full 2x2x2 factorial {argmax, sampled} x {KL off/on} x {InfoNCE off/on} + baselines + controls
    METHODS: tuple = ("vanilla", "aug_det", "det_kl", "augcon_det", "sb_det",
                      "aug_sample", "sb_noCon", "sb_noKL", "scriptbridge",
                      "augcon_uroman", "rdrop", "charnoise", "upper_dual")
    REDUCED_METHODS: tuple = ("vanilla", "aug_det", "aug_sample", "augcon_det", "sb_det", "scriptbridge")
    SEEDS: tuple = (13, 42, 2026, 7, 101)
    REDUCED_SEEDS: tuple = (13, 42, 2026)
    RESULTS_DIR: str = "/content/scriptbridge_results"
    SAVE_TO_DRIVE: bool = False

cfg = CFG()
if cfg.QUICK_RUN:
    cfg.EPOCHS, cfg.SEEDS, cfg.REDUCED_SEEDS, cfg.TASKS, cfg.AUTO_TUNE = 2, (13, 42), (13,), ("myc",), False
    cfg.MODELS, cfg.MAX_MODELS, cfg.RUN_BYT5, cfg.N_BOOT = ("distilbert-base-multilingual-cased",), 1, False, 200
if cfg.SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    cfg.RESULTS_DIR = "/content/drive/MyDrive/scriptbridge_results"
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print(json.dumps(asdict(cfg), indent=1))

## 4. Learned stochastic transliterator `T_θ`
`CAND` only defines the **search space** of possible spellings per Arabic unit, i.e. which Latin strings an Arabic letter *may* map to. All probabilities are **learned** by hard-EM Viterbi alignment on DoDa word pairs. Probabilities start uniform, so nothing is hand-tuned.

In [ ]:
# ===== Learned stochastic Darija transliterator (Arabic <-> Arabizi), trained by hard-EM on DoDa =====
import re, math, random, unicodedata
from collections import defaultdict, Counter

SHADDA = "ّ"
AR_DIAC_RE = re.compile("[ً-ِْـ]")       # harakat, tanween, sukun, tatweel (not shadda)
AR_PUNCT = str.maketrans({"،": ",", "؟": "?", "؛": ";"})
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
VOWELS = "aeiou"
AR_LETTER = re.compile("[ء-يٱ-ۓڠ-گ]")

# Candidate inventory = linguistic search space only. Probabilities are LEARNED from data.
CAND = {
 "ب": "b p", "ت": "t tt", "ث": "t th", "ج": "j g dj", "ح": "7 h", "خ": "kh 5 x", "د": "d",
 "ذ": "d dh z", "ر": "r", "ز": "z", "س": "s ss", "ش": "ch sh x", "ص": "s S", "ض": "d D",
 "ط": "t T 6", "ظ": "d z D", "ع": "3 a e", "غ": "gh 4 8 r", "ف": "f", "ق": "9 q k g",
 "ك": "k c", "ل": "l", "م": "m", "ن": "n", "ه": "h", "ة": "a e ah at", "ى": "a i",
 "ا": "a e i o _", "أ": "a 2a 2 e o _", "إ": "i e 2i _", "آ": "a 2a", "ء": "2 _ a e",
 "ئ": "2 y i e _", "ؤ": "2 o ou _", "و": "w ou o u oo _", "ي": "y i ee ii e _",
 "گ": "g", "ڭ": "g", "ڤ": "v f", "پ": "p", "ال": "l el al _ e", SHADDA: "@dup _",
}
CAND = {k: [("" if o == "_" else o) for o in v.split()] for k, v in CAND.items()}

def norm_ar(t):
    t = unicodedata.normalize("NFC", t).translate(AR_DIGITS).translate(AR_PUNCT)
    return AR_DIAC_RE.sub("", t)

def ar_units(word):
    """split an Arabic word into units; 'ال' after optional proclitic is one unit"""
    u, i = [], 0
    if len(word) > 3 and word[0] in "وبفكل" and word[1:3] == "ال": u, i = [word[0], "ال"], 3
    elif len(word) > 2 and word[:2] == "ال": u, i = ["ال"], 2
    u += list(word[i:])
    return u

def pos_tag(i, n): return "I" if i == 0 else ("F" if i == n - 1 else "M")

class DarijaTranslit:
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.p = {}                              # (unit,pos) -> {opt: prob}
        self.p_ep, self.ep_vowel = 0.05, {v: .2 for v in VOWELS}
        self.rev = {}                            # (lat_unit,pos) -> {ar: prob}
        self.seg = {}                            # lat_unit -> logprob
        self.temp = {"ar2lat": 1.0, "lat2ar": 1.0} # sampling temperature per direction (selected on DoDa dev)
        for k, opts in CAND.items():
            for ps in "IMF": self.p[(k, ps)] = {o: 1 / len(opts) for o in opts}

    # ---------- alignment (Viterbi) ----------
    def _align(self, units, lat):
        n, m = len(units), len(lat)
        NEG = -1e18
        dp = [[NEG] * (m + 1) for _ in range(n + 1)]; bp = [[None] * (m + 1) for _ in range(n + 1)]
        dp[0][0] = 0.0
        lep, lnoep = math.log(self.p_ep), math.log(1 - self.p_ep)
        for i in range(n + 1):
            for j in range(m + 1):
                s = dp[i][j]
                if s <= NEG / 2: continue
                # epenthetic vowel (only after first unit, before last char consumed)
                if j < m and lat[j] in VOWELS and i > 0:
                    v = s + lep + math.log(self.ep_vowel[lat[j]])
                    if v > dp[i][j + 1]: dp[i][j + 1], bp[i][j + 1] = v, (i, j, "EP", lat[j])
                if i == n: continue
                u = units[i]; ps = pos_tag(i, n)
                dist = self.p.get((u, ps))
                if dist is None: continue
                for o, pr in dist.items():
                    if o == "@dup":
                        if j == 0: continue
                        o_str = lat[j - 1]
                    else: o_str = o
                    L = len(o_str)
                    if lat[j:j + L] == o_str:
                        v = s + math.log(pr) + lnoep
                        if v > dp[i + 1][j + L]: dp[i + 1][j + L], bp[i + 1][j + L] = v, (i, j, u, o, ps, o_str)
        if dp[n][m] <= NEG / 2: return None
        path, i, j = [], n, m
        while (i, j) != (0, 0):
            b = bp[i][j]; path.append(b); i, j = b[0], b[1]
        return path[::-1]

    def fit(self, pairs, iters=3, max_words=80000, seed=0, verbose=True):
        """pairs: list of (arabic_sentence, latin_sentence)"""
        rng = random.Random(seed); wp = []
        for ar, la in pairs:
            aw = [w for w in re.findall(r"\S+", norm_ar(ar)) if AR_LETTER.search(w)]
            lw = [w for w in re.findall(r"\S+", la.lower()) if re.search("[a-z]", w)]
            aw = [re.sub(r"[^ء-ۿ" + SHADDA + "]", "", w) for w in aw]
            lw = [re.sub(r"[^a-z0-9]", "", w) for w in lw]
            if len(aw) == len(lw): wp += [(a, l) for a, l in zip(aw, lw) if a and l]
        rng.shuffle(wp); wp = wp[:max_words]
        for it in range(iters):
            cnt = defaultdict(Counter); rcnt = defaultdict(Counter); seg = Counter()
            ep, opp, epv, ok = 0, 0, Counter(), 0
            for a, l in wp:
                units = ar_units(a); path = self._align(units, l)
                if path is None: continue
                ok += 1
                n = len(units)
                for st in path:
                    if st[2] == "EP":
                        ep += 1; epv[st[3]] += 1; rcnt[(st[3], "M")][""] += 1; seg[st[3]] += 1
                    else:
                        _, _, u, o, ps, o_str = st; opp += 1
                        cnt[(u, ps)][o] += 1
                        if o_str:
                            rcnt[(o_str, ps)][u if o != "@dup" else SHADDA] += 1; seg[o_str] += 1
            for key, opts in self.p.items():
                c = cnt.get(key, Counter()); tot = sum(c[o] for o in opts) + self.alpha * len(opts)
                self.p[key] = {o: (c[o] + self.alpha) / tot for o in opts}
            self.p_ep = max(1e-3, ep / max(1, ep + opp))
            tv = sum(epv.values()) + 5 * self.alpha
            self.ep_vowel = {v: (epv[v] + self.alpha) / tv for v in VOWELS}
            if verbose: print(f"  EM iter {it}: aligned {ok}/{len(wp)} word pairs, p_epenthesis={self.p_ep:.3f}")
        # reverse model P(ar | latin unit, pos) and unigram segmentation model
        self.rev = {}
        for k, c in rcnt.items():
            t = sum(c.values()); self.rev[k] = {a: v / t for a, v in c.items() if v / t > 0.01}
        tot = sum(seg.values())
        self.seg = {u: math.log(c / tot) for u, c in seg.items() if c >= 3}
        return self

    # ---------- generation ----------
    def _choose(self, dist, rng, stochastic, temp=1.0):
        if not stochastic or temp <= 0: return max(dist, key=dist.get)
        keys = list(dist); w = [dist[k] ** (1.0 / temp) for k in keys]
        return rng.choices(keys, weights=w, k=1)[0]

    def ar2lat(self, text, rng=random, stochastic=True):
        out = []
        for tok in re.split(r"(\s+)", norm_ar(text)):
            if not tok or tok.isspace() or not AR_LETTER.search(tok): out.append(tok); continue
            units = ar_units(tok); n = len(units); buf = []; tmp = self.temp["ar2lat"]
            for i, u in enumerate(units):
                d = self.p.get((u, pos_tag(i, n)))
                if d is None: buf.append(u); continue          # emoji, latin, digits, punctuation
                o = self._choose(d, rng, stochastic, tmp)
                if o == "@dup": o = buf[-1][-1] if buf and buf[-1] else ""
                buf.append(o)
                if stochastic and i < n - 1 and rng.random() < self.p_ep * min(1.0, tmp):
                    buf.append(self._choose(self.ep_vowel, rng, True))
            out.append("".join(buf))
        return "".join(out)

    def _segment(self, w):
        n = len(w); best = [(-1e18, None)] * (n + 1); best[0] = (0.0, None)
        for j in range(1, n + 1):
            for L in (1, 2, 3):
                i = j - L
                if i < 0: break
                u = w[i:j]; s = self.seg.get(u, -30.0 if L == 1 else None)
                if s is None or best[i][0] <= -1e17: continue
                if best[i][0] + s > best[j][0]: best[j] = (best[i][0] + s, i)
        segs, j = [], n
        while j > 0:
            i = best[j][1]; segs.append(w[i:j]); j = i
        return segs[::-1]

    def lat2ar(self, text, rng=random, stochastic=True):
        out = []
        for tok in re.split(r"(\s+)", text):
            low = tok.lower()
            if not tok or tok.isspace() or not re.search("[a-z]", low): out.append(tok); continue
            core = re.match(r"^([^a-z0-9]*)([a-z0-9]+)([^a-z0-9]*)$", low)
            if not core: out.append(tok); continue
            pre, w, post = core.groups()
            w = re.sub(r"([a-z])\1{2,}", r"\1\1", w)
            segs = self._segment(w); n = len(segs); buf = []
            for i, s in enumerate(segs):
                ps = pos_tag(i, n)
                d = self.rev.get((s, ps)) or self.rev.get((s, "M")) or self.rev.get((s, "I")) or self.rev.get((s, "F"))
                if not d: buf.append(s); continue
                buf.append(self._choose(d, rng, stochastic, self.temp["lat2ar"]))
            out.append(pre + "".join(buf) + post)
        return "".join(out)

In [ ]:
doda = pd.read_csv(cfg.DODA_URL).dropna(subset=["darija", "darija_ar"])
doda = doda.sample(frac=1, random_state=0).reset_index(drop=True)
_a, _b = cfg.TR_TEST_PAIRS, cfg.TR_TEST_PAIRS + cfg.TR_DEV_PAIRS
doda_test, doda_dev, doda_train = doda.iloc[:_a], doda.iloc[_a:_b], doda.iloc[_b:]
print("DoDa pairs:", len(doda), "| train", len(doda_train), "| dev", len(doda_dev), "| test", len(doda_test))
t0 = time.time()
T = DarijaTranslit().fit(list(zip(doda_train.darija_ar, doda_train.darija)),
                         iters=cfg.TR_EM_ITERS, max_words=cfg.TR_MAX_WORDS)
print(f"fitted in {time.time()-t0:.1f}s")
json.dump({"p": {f"{u}|{ps}": d for (u, ps), d in T.p.items()}, "p_ep": T.p_ep, "ep_vowel": T.ep_vowel,
           "rev": {f"{u}|{ps}": d for (u, ps), d in T.rev.items()}, "seg": T.seg},
          open(os.path.join(cfg.RESULTS_DIR, "transliterator_params.json"), "w"), ensure_ascii=False)

import uroman as _ur
UROMAN = _ur.Uroman()
def uroman(text): return UROMAN.romanize_string(text)

def transliterate(text, src_script, rng=random, stochastic=True):
    return T.ar2lat(text, rng, stochastic) if src_script == "arabic" else T.lat2ar(text, rng, stochastic)

rng = random.Random(0)
for s in ["الله يعطيك الصحة خويا، زوين بزاف", "هادشي ماشي مزيان، حشومة عليكم"]:
    print(s, "\n  argmax:", T.ar2lat(s, stochastic=False), "\n  uroman:", uroman(s))
    for _ in range(3): print("  sample:", T.ar2lat(s, rng))

### 4.1 Automatic intrinsic evaluation of `T_θ` (held-out DoDa): CER for argmax, one sample and best-of-k, vs uroman

In [ ]:
def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]
def _n(t): return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", t.lower())).strip()
def cer(h, r): h, r = _n(h), _n(r); return lev(h, r) / max(1, len(r))

K = 10; rng = random.Random(1); sub = doda_test.iloc[:1000]
rows = []
for name, src, ref, fwd in [("ar→lat", sub.darija_ar, sub.darija, T.ar2lat), ("lat→ar", sub.darija, sub.darija_ar, T.lat2ar)]:
    rows.append(dict(direction=name, system="T_θ argmax", CER=np.mean([cer(fwd(s, stochastic=False), r) for s, r in zip(src, ref)])))
    rows.append(dict(direction=name, system="T_θ 1 sample", CER=np.mean([cer(fwd(s, rng, True), r) for s, r in zip(src, ref)])))
    rows.append(dict(direction=name, system=f"T_θ best-of-{K}", CER=np.mean([min(cer(fwd(s, rng, True), r) for _ in range(K)) for s, r in zip(src, ref)])))
rows.append(dict(direction="ar→lat", system="uroman", CER=np.mean([cer(uroman(s), r) for s, r in zip(sub.darija_ar, sub.darija)])))
tr_table = pd.DataFrame(rows); tr_table["CER"] = (100 * tr_table.CER).round(2)
display(tr_table)
tr_table.to_latex(os.path.join(cfg.RESULTS_DIR, "table_transliterator_cer.tex"), index=False)

### 4.2 Automatic sampling-temperature selection (label-free, task-free)
Samples are drawn with probabilities ∝ p^(1/τ), and epenthesis probability is scaled by min(1, τ). For each direction, τ is chosen on **DoDa dev** (never on task data) as the value whose samples are closest to real target-script text in character-trigram distribution (Jensen–Shannon divergence). τ = 0 (argmax) is a legitimate candidate: if it wins, the 'sampled' methods become deterministic and the paper reports that.

In [ ]:
from collections import Counter
def ngram_dist(texts, n=3):
    c = Counter()
    for t in texts:
        t = f" {_n(t)} "; c.update(t[i:i + n] for i in range(len(t) - n + 1))
    s = sum(c.values()); return {k: v / s for k, v in c.items()}
def jsd(p, q):
    keys = set(p) | set(q); m = {k: .5 * (p.get(k, 0) + q.get(k, 0)) for k in keys}
    kl = lambda a: sum(a[k] * math.log(a[k] / m[k]) for k in a if a[k] > 0)
    return .5 * kl(p) + .5 * kl(q)
temp_rows = []
for dirn, src_txt, ref_txt, fn in [("ar2lat", doda_dev.darija_ar, doda_dev.darija, T.ar2lat),
                                   ("lat2ar", doda_dev.darija, doda_dev.darija_ar, T.lat2ar)]:
    real = ngram_dist(ref_txt); best = (1e9, 1.0)
    for tau in tuple(cfg.TEMP_GRID):
        T.temp[dirn] = tau; r_ = random.Random(0)
        j = jsd(ngram_dist([fn(t, r_, tau > 0) for t in src_txt]), real)
        temp_rows.append(dict(direction=dirn, tau=tau, JSD=round(j, 4)))
        if tau in cfg.TEMP_GRID and j < best[0]: best = (j, tau)
    T.temp[dirn] = best[1] if cfg.AUTO_TEMP else 1.0
temp_tab = pd.DataFrame(temp_rows).pivot(index="tau", columns="direction", values="JSD"); display(temp_tab)
temp_tab.to_latex(os.path.join(cfg.RESULTS_DIR, "table_temperature.tex"))
print("Selected temperatures:", T.temp)

### 4.3 Multi-reference evaluation on DoDa's spelling-variant lexicon
DoDa's word lists give two to six human Latin spellings for thousands of words, so the transliterator can be evaluated against several valid answers without human judges. The cell reports the accuracy of the argmax output, the share of k samples that are valid human spellings (Precision@k), the share of a word's human spellings that the sampler produces (Coverage@k), whether at least one sample is valid (Any@k), and, for the Latin to Arabic direction, whether every human Latin variant maps back to the Arabic form.

All words and unseen words are reported separately. An unseen word's Arabic form never occurs in the transliterator's training sentences. The cell also reports CER at the selected temperature for one sample, which is the quality the classifier actually sees.

In [ ]:
import urllib.parse, requests
LEX_FILES = {"semantic categories": ["animals","art","clothes","colors","economy","education","emotions","environment","family",
                                     "femalenames","food","health","humanbody","malenames","numbers","places","plants",
                                     "professions","religion","sport","time"],
             "syntactic categories": ["(in)definite","adjectives","adverbs","conjug_past","conjug_present","imperatives",
                                      "masculine_feminine_plural","nouns","prepositions","pronouns","verb-to-noun","verbs"]}
BASE = "https://raw.githubusercontent.com/darija-open-dataset/dataset/main/"
lex_rows = []
for folder, names in LEX_FILES.items():
    for nm in names:
        try:
            d_ = pd.read_csv(BASE + urllib.parse.quote(f"{folder}/{nm}.csv"))
        except Exception: continue
        vcols = [c for c in d_.columns if re.fullmatch(r"n\d+", str(c))]
        if "darija_ar" not in d_.columns or not vcols: continue
        for _, r_ in d_.iterrows():
            ar = str(r_["darija_ar"]).strip() if pd.notna(r_["darija_ar"]) else ""
            refs = sorted({_n(str(r_[c])) for c in vcols if pd.notna(r_[c]) and _n(str(r_[c]))})
            if ar and len(refs) >= 2 and len(ar.split()) == 1: lex_rows.append((ar, refs))
train_vocab = set(w for t in doda_train.darija_ar for w in re.findall(r"\S+", norm_ar(t)))
lex = pd.DataFrame(lex_rows, columns=["ar", "refs"]).drop_duplicates("ar")
lex["unseen"] = ~lex.ar.map(lambda a: norm_ar(a) in train_vocab)
print(f"lexicon entries with >=2 human spellings: {len(lex)} (unseen: {int(lex.unseen.sum())})")

def lexicon_eval(sub, taus=(0.0, None, 1.0), k=cfg.LEXICON_K):
    out = []; saved = T.temp["ar2lat"]
    for tau in taus:
        tau_v = saved if tau is None else tau; T.temp["ar2lat"] = tau_v; r_ = random.Random(0)
        acc = prec = cov = anyk = 0.0
        for ar, refs in zip(sub.ar, sub.refs):
            R = set(refs)
            if tau_v == 0:
                o = _n(T.ar2lat(ar, stochastic=False)); samp = [o] * k
            else:
                samp = [_n(T.ar2lat(ar, r_, True)) for _ in range(k)]
            acc += samp[0] in R; prec += np.mean([x in R for x in samp])
            cov += len(R & set(samp)) / len(R); anyk += any(x in R for x in samp)
        n = max(1, len(sub))
        out.append(dict(system=f"T_θ τ={tau_v:g}" + (" (selected)" if tau is None else ""), acc=100*acc/n,
                        precision_at_k=100*prec/n, coverage_at_k=100*cov/n, any_at_k=100*anyk/n))
    T.temp["ar2lat"] = saved
    acc_u = np.mean([_n(uroman(a)) in set(r) for a, r in zip(sub.ar, sub.refs)])
    out.append(dict(system="uroman", acc=100*acc_u, precision_at_k=100*acc_u, coverage_at_k=np.nan, any_at_k=100*acc_u))
    # Latin -> Arabic robustness across human spelling variants
    l2a = [norm_ar(T.lat2ar(v, stochastic=False)).replace(" ", "") == norm_ar(a).replace(" ", "") for a, refs in zip(sub.ar, sub.refs) for v in refs]
    out.append(dict(system="T_θ lat→ar argmax, all variants", acc=100*np.mean(l2a), precision_at_k=np.nan, coverage_at_k=np.nan, any_at_k=np.nan))
    return pd.DataFrame(out)

lex_tabs = []
for name, sub_ in [("all", lex), ("unseen", lex[lex.unseen])]:
    t_ = lexicon_eval(sub_); t_.insert(0, "words", f"{name} (n={len(sub_)})"); lex_tabs.append(t_)
lex_table = pd.concat(lex_tabs).round(2); display(lex_table)
lex_table.to_latex(os.path.join(cfg.RESULTS_DIR, "table_lexicon_multiref.tex"), index=False)

# CER of the deployed sampler (selected tau, one sample) on held-out DoDa sentences
rng = random.Random(2); sub = doda_test.iloc[:1000]
cer_sel = [dict(direction="ar→lat", system=f"T_θ 1 sample @ selected τ={T.temp['ar2lat']:g}",
                CER=100*np.mean([cer(T.ar2lat(s, rng, T.temp['ar2lat'] > 0), r) for s, r in zip(sub.darija_ar, sub.darija)])),
           dict(direction="lat→ar", system=f"T_θ 1 sample @ selected τ={T.temp['lat2ar']:g}",
                CER=100*np.mean([cer(T.lat2ar(s, rng, T.temp['lat2ar'] > 0), r) for s, r in zip(sub.darija, sub.darija_ar)]))]
tr_table = pd.concat([tr_table, pd.DataFrame(cer_sel).round(2)], ignore_index=True); display(tr_table)
tr_table.to_latex(os.path.join(cfg.RESULTS_DIR, "table_transliterator_cer.tex"), index=False)

### 4.4 Same-data neural transliteration baseline (ByT5-small)
The Moroccan-specific transliterators in the literature do not release code, and their published numbers come from different data. We therefore fine-tune a byte-level seq2seq model on **exactly the same DoDa training pairs**, in both directions (task prefix), and evaluate it on the same held-out pairs. Set `RUN_BYT5=False` to skip.

In [ ]:
def load_byt5(name):
    from transformers import AutoTokenizer as _AT, AutoModelForSeq2SeqLM
    return _AT.from_pretrained(name), AutoModelForSeq2SeqLM.from_pretrained(name)

BYT5_RESULTS = None
if cfg.RUN_BYT5:
    _dev = "cuda" if torch.cuda.is_available() else "cpu"
    btok, bmodel = load_byt5(cfg.BYT5_NAME); bmodel.to(_dev)
    pairs = [("ar2lat: " + a, l) for a, l in zip(doda_train.darija_ar, doda_train.darija)] + \
            [("lat2ar: " + l, a) for a, l in zip(doda_train.darija_ar, doda_train.darija)]
    random.Random(0).shuffle(pairs)
    opt_b = torch.optim.AdamW(bmodel.parameters(), lr=cfg.BYT5_LR)
    steps_b = math.ceil(len(pairs) / cfg.BYT5_BATCH) * cfg.BYT5_EPOCHS
    from transformers import get_linear_schedule_with_warmup as _sched
    sch_b = _sched(opt_b, int(0.05 * steps_b), steps_b); bmodel.train(); t0 = time.time()
    _amp = torch.bfloat16 if (_dev == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
    for ep in range(cfg.BYT5_EPOCHS):
        for i in range(0, len(pairs), cfg.BYT5_BATCH):
            src_b, tgt_b = zip(*pairs[i:i + cfg.BYT5_BATCH])
            enc = btok(list(src_b), padding=True, truncation=True, max_length=cfg.BYT5_MAXLEN, return_tensors="pt").to(_dev)
            lab = btok(list(tgt_b), padding=True, truncation=True, max_length=cfg.BYT5_MAXLEN, return_tensors="pt").input_ids.to(_dev)
            lab[lab == btok.pad_token_id] = -100
            with torch.autocast(device_type="cuda", dtype=_amp, enabled=_dev == "cuda"):
                loss = bmodel(**enc, labels=lab).loss
            opt_b.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(bmodel.parameters(), 1.0); opt_b.step(); sch_b.step()
            if (i // cfg.BYT5_BATCH) % 500 == 0: print(f"  ByT5 step {i // cfg.BYT5_BATCH}/{steps_b} loss={loss.item():.3f} ({time.time()-t0:.0f}s)")
    bmodel.eval()
    @torch.no_grad()
    def byt5_translit(texts, prefix, bs=64):
        out = []
        for i in range(0, len(texts), bs):
            enc = btok([prefix + t for t in texts[i:i + bs]], padding=True, truncation=True, max_length=cfg.BYT5_MAXLEN, return_tensors="pt").to(_dev)
            gen = bmodel.generate(**enc, max_new_tokens=cfg.BYT5_MAXLEN, num_beams=1)
            out += btok.batch_decode(gen, skip_special_tokens=True)
        return out
    sub = doda_test.iloc[:1000]
    h1 = byt5_translit(sub.darija_ar.tolist(), "ar2lat: "); h2 = byt5_translit(sub.darija.tolist(), "lat2ar: ")
    BYT5_RESULTS = pd.DataFrame([dict(direction="ar→lat", system="ByT5-small (same data)", CER=100*np.mean([cer(h, r) for h, r in zip(h1, sub.darija)])),
                                 dict(direction="lat→ar", system="ByT5-small (same data)", CER=100*np.mean([cer(h, r) for h, r in zip(h2, sub.darija_ar)]))]).round(2)
    print(f"ByT5 trained in {time.time()-t0:.0f}s"); display(BYT5_RESULTS)
    tr_table = pd.concat([tr_table, BYT5_RESULTS], ignore_index=True); display(tr_table)
    tr_table.to_latex(os.path.join(cfg.RESULTS_DIR, "table_transliterator_cer.tex"), index=False)
    del bmodel; torch.cuda.empty_cache()

## 5. Load task data automatically

In [ ]:
def detect_script(text):
    ar = sum(1 for c in text if "؀" <= c <= "ۿ" or "ݐ" <= c <= "ݿ")
    la = sum(1 for c in text if "a" <= c.lower() <= "z")
    if ar == 0 and la == 0: return "other"
    return "arabic" if ar >= la else "latin"

def load_myc():
    import requests
    raw = requests.get(cfg.MYC_URL, timeout=60).content
    for enc in ("utf-16", "utf-8-sig", "utf-8"):
        try: d = pd.read_csv(io.BytesIO(raw), encoding=enc); break
        except Exception: continue
    d = d.rename(columns={"sentence": "text", "polarity": "label"})
    d["label"] = d["label"].map({1: "positive", -1: "negative"}).fillna(d["label"].astype(str))
    return d[["text", "label"]]

def load_offensive():
    import requests
    files = requests.get(cfg.OFFENSIVE_MENDELEY, timeout=60, headers={"Accept": "application/json"}).json()
    f = next(x for x in files if x["filename"].lower().endswith((".csv", ".xlsx")))
    raw = requests.get(f["content_details"]["download_url"], timeout=120).content
    d = pd.read_csv(io.BytesIO(raw)) if f["filename"].lower().endswith(".csv") else pd.read_excel(io.BytesIO(raw))
    obj = [c for c in d.columns if pd.api.types.is_object_dtype(d[c]) or pd.api.types.is_string_dtype(d[c])]
    tcol = max(obj, key=lambda c: d[c].astype(str).str.len().mean())
    lcol = next(c for c in d.columns if c != tcol and d[c].nunique() <= 5)
    return d[[tcol, lcol]].rename(columns={tcol: "text", lcol: "label"})

EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2300-\u23FF\u2B00-\u2BFF\uFE0F\u200D\U000E0000-\U000E007F]")
def clean_text(t, strip_emoji):
    t = re.sub(r"<[^>]+>|https?://\S+|&\w+;", " ", str(t)).replace('"', " ")
    if strip_emoji: t = EMOJI_RE.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

AR_CHARS = re.compile("[\u0600-\u06FF\u0750-\u077F]"); LA_CHARS = re.compile("[A-Za-z]")
def is_mixed(t):
    a, l = len(AR_CHARS.findall(t)), len(LA_CHARS.findall(t))
    return min(a, l) >= 3 and min(a, l) / max(1, a + l) >= 0.15
def strip_other_script(t):
    s_ = detect_script(t)
    t = LA_CHARS.sub(" ", t) if s_ == "arabic" else (AR_CHARS.sub(" ", t) if s_ == "latin" else t)
    return re.sub(r"\s+", " ", t).strip()

def prepare(d, seed=1234, strip_emoji=None, strip_other=False):
    strip_emoji = cfg.STRIP_EMOJI if strip_emoji is None else strip_emoji
    d = d.dropna().copy()
    d["raw"] = d["text"].astype(str)
    d["text"] = d["raw"].map(lambda t: clean_text(t, strip_emoji))
    if strip_other: d["text"] = d.text.map(strip_other_script)
    d = d[(d.text.str.len() >= cfg.MIN_CHARS) & (d.text.str.split().str.len() >= cfg.MIN_TOKENS)].drop_duplicates("text")
    d["script"] = d.text.map(detect_script); d = d[d.script.isin(["arabic", "latin"])]
    d["mixed"] = d.text.map(is_mixed)
    if cfg.QUICK_RUN:
        d = pd.concat([d[d.script == s].sample(min((d.script == s).sum(), 1500), random_state=0) for s in ["arabic", "latin"]])
    labels = sorted(d.label.astype(str).unique()); d["y"] = d.label.astype(str).map({l: i for i, l in enumerate(labels)})
    from sklearn.model_selection import train_test_split
    splits = {}
    for s in ["arabic", "latin"]:
        g = d[d.script == s]
        tr, te = train_test_split(g, test_size=cfg.TEST_FRAC, stratify=g.y, random_state=seed)
        tr, dv = train_test_split(tr, test_size=cfg.DEV_FRAC / (1 - cfg.TEST_FRAC), stratify=tr.y, random_state=seed)
        splits[s] = {k: v.reset_index(drop=True) for k, v in zip(["train", "dev", "test"], [tr, dv, te])}
    return d, labels, splits

TASK_DATA, RAW_TASK = {}, {}
for task in cfg.TASKS:
    try:
        raw = {"myc": load_myc, "offensive": load_offensive}[task]()
        d, labels, splits = prepare(raw)
        TASK_DATA[task] = (labels, splits); RAW_TASK[task] = raw
        print(f"== {task}: {len(d)} rows, labels={labels}, mixed-script rows: {int(d.mixed.sum())}")
        print(d.groupby(["script", "label"]).size().unstack(fill_value=0))
    except Exception as e:
        print(f"!! task '{task}' skipped automatically ({type(e).__name__}: {e})")

### 5.1 Automatic shortcut, contamination and duplicate audit
All checks are model-free and run for every task and direction.

1. An emoji-only classifier is trained on raw text in one script and tested on the other, and emoji prevalence is reported per class and script.
2. A character n-gram classifier is evaluated in-script, cross-script, and cross-script after test-time transliteration.
3. Majority-class and length, punctuation and digit baselines are computed, together with label prevalence per script.
4. For contamination, the cell lists the most influential n-grams that fire on the other script, counts mixed-script items and source items containing the other script's letters, and reruns the classifier after stripping the other script's letters from all texts and after dropping mixed-script items.
5. Near-duplicates are target test items whose transliteration into the source script has a near-identical source training neighbour (character-trigram cosine of at least 0.9).

A direction is flagged as contaminated when its character n-gram cross-script AUROC reaches `CONTAM_AUROC`. If stripping the other script's letters removes the cue, a cleaned copy of the task is added and run with the reduced grid.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.metrics import f1_score as _f1, roc_auc_score as _auc
from sklearn.neighbors import NearestNeighbors
def _emo(texts): return [" ".join(EMOJI_RE.findall(t)) or "NONE" for t in texts]
def charlr(tr, te_list):
    v2 = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    c2 = _LR(max_iter=3000, C=4).fit(v2.fit_transform(tr.text), tr.y)
    res = []
    for te in te_list:
        P = c2.predict_proba(v2.transform(te.text))
        f = _f1(te.y, P.argmax(1), average="macro")
        a = _auc(te.y, P[:, 1]) if P.shape[1] == 2 and te.y.nunique() == 2 else np.nan
        res.append((100 * f, 100 * a))
    return res, v2, c2

audit, emo_rows, feat_rows, CONTAMINATED, CLEAN_TASKS = [], [], [], set(), {}
for task, raw in list(RAW_TASK.items()):
    r_ = raw.dropna().copy(); r_["text"] = r_["text"].astype(str)
    r_["script"] = r_.text.map(lambda t: detect_script(clean_text(t, True)))
    r_ = r_[r_.script.isin(["arabic", "latin"])]
    for (sc, lab), g in r_.groupby(["script", "label"]):
        emo_rows.append(dict(task=task, script=sc, label=lab, n=len(g), pct_with_emoji=100 * g.text.map(lambda t: bool(EMOJI_RE.search(t))).mean()))
    LABELS_, sp_ = TASK_DATA[task]
    for src, tgt in [("arabic", "latin"), ("latin", "arabic")]:
        A, B = r_[r_.script == src], r_[r_.script == tgt]
        v = CountVectorizer(token_pattern=r"\S+"); c = _LR(max_iter=3000).fit(v.fit_transform(_emo(A.text)), A.label.astype(str))
        emo_f1 = _f1(B.label.astype(str), c.predict(v.transform(_emo(B.text))), average="macro")
        tr, te_in, te_x = sp_[src]["train"], sp_[src]["test"], sp_[tgt]["test"]
        ttn = te_x.assign(text=[transliterate(t, tgt, stochastic=False) for t in te_x.text])
        (in_, cross_, ttn_), v2, c2 = charlr(tr, [te_in, te_x, ttn])
        # influential n-grams that fire on the target script
        Xt = v2.transform(te_x.text); active = np.asarray((Xt > 0).sum(0)).ravel()
        vocab = np.array(v2.get_feature_names_out()); w = c2.coef_[0] if c2.coef_.shape[0] == 1 else c2.coef_.max(0)
        score = np.abs(w) * active
        for j in np.argsort(-score)[:15]:
            feat_rows.append(dict(task=task, direction=f"{src}→{tgt}", ngram=repr(vocab[j]), weight=round(float(w[j]), 3), n_target_docs=int(active[j])))
        # interventions
        def strip(df):
            df = df.assign(text=df.text.map(strip_other_script)); return df[df.text.str.len() > 0]
        res_s, _, _ = charlr(strip(tr), [strip(te_in), strip(te_x)]); cross_strip = res_s[1]
        nomix = lambda df: df[~df.mixed]
        (_, cross_nomix), _, _ = charlr(nomix(tr), [nomix(te_in), nomix(te_x)])
        other = LA_CHARS if src == "arabic" else AR_CHARS
        # cross-script near-duplicates
        vv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3)).fit(list(tr.text) + list(ttn.text))
        nn_ = NearestNeighbors(n_neighbors=1, metric="cosine").fit(vv.transform(tr.text))
        dist, _ = nn_.kneighbors(vv.transform(ttn.text))
        Lf = lambda d_: np.c_[np.log1p(d_.text.str.len()), np.log1p(d_.text.str.split().str.len()),
                              d_.text.str.count(r"[!?]"), d_.text.str.count(r"\d")]
        c3 = _LR(max_iter=3000).fit(Lf(tr), tr.y); maj = tr.y.mode()[0]
        row = dict(task=task, direction=f"{src}→{tgt}", pos_rate_src=100 * (tr.y == 1).mean(), pos_rate_tgt=100 * (te_x.y == 1).mean(),
                   emoji_only_cross_F1=100 * emo_f1,
                   majority_cross_F1=100 * _f1(te_x.y, np.full(len(te_x), maj), average="macro"),
                   length_only_cross_F1=100 * _f1(te_x.y, c3.predict(Lf(te_x)), average="macro"),
                   charLR_in=in_[0], charLR_cross=cross_[0], charLR_cross_TTN=ttn_[0], charLR_cross_AUROC=cross_[1],
                   pct_src_train_with_other_letters=100 * tr.text.map(lambda t: bool(other.search(t))).mean(),
                   pct_mixed_src=100 * tr.mixed.mean(), pct_mixed_tgt=100 * te_x.mixed.mean(),
                   AUROC_after_strip=cross_strip[1], AUROC_without_mixed=cross_nomix[1],
                   near_dup_tgt_items=int((dist.ravel() <= 0.1).sum()))
        row["contaminated"] = bool(cross_[1] >= 100 * cfg.CONTAM_AUROC)
        if row["contaminated"]:
            CONTAMINATED.add((task, src))
            if cfg.RUN_CLEANED and cross_strip[1] < 100 * cfg.CONTAM_AUROC: CLEAN_TASKS[f"{task}_clean"] = task
        audit.append(row)
audit = pd.DataFrame(audit).round(2)
with pd.option_context("display.width", 300, "display.max_columns", 40): display(audit.T)
display(pd.DataFrame(emo_rows).round(1)); display(pd.DataFrame(feat_rows))
audit.to_csv(os.path.join(cfg.RESULTS_DIR, "audit.csv"), index=False)
pd.DataFrame(feat_rows).to_csv(os.path.join(cfg.RESULTS_DIR, "audit_top_ngrams.csv"), index=False)
pd.DataFrame(emo_rows).to_csv(os.path.join(cfg.RESULTS_DIR, "audit_emoji_by_class.csv"), index=False)
audit.to_latex(os.path.join(cfg.RESULTS_DIR, "table_shortcut_gap_audit.tex"), index=False)
print("Contaminated (task, source script):", CONTAMINATED)
for new, base in CLEAN_TASKS.items():
    d_, labels_, splits_ = prepare(RAW_TASK[base], strip_other=True)
    TASK_DATA[new] = (labels_, splits_); print(f"added cleaned task '{new}' ({len(d_)} rows, other-script letters stripped)")

### 5.2 Label-free distributional check: does sampled Arabizi look like real MYC Arabizi? (char-trigram Jensen–Shannon divergence, train splits only)

In [ ]:
from collections import Counter
def ngram_dist(texts, n=3):
    c = Counter()
    for t in texts:
        t = f" {_n(t)} "; c.update(t[i:i + n] for i in range(len(t) - n + 1))
    s = sum(c.values()); return {k: v / s for k, v in c.items()}
def jsd(p, q):
    keys = set(p) | set(q); m = {k: .5 * (p.get(k, 0) + q.get(k, 0)) for k in keys}
    kl = lambda a: sum(a[k] * math.log(a[k] / m[k]) for k in a if a[k] > 0)
    return .5 * kl(p) + .5 * kl(q)
_, sp = TASK_DATA["myc"]
ar_tr, la_tr = sp["arabic"]["train"].text.tolist()[:4000], sp["latin"]["train"].text.tolist()[:4000]
real = ngram_dist(la_tr); rng = random.Random(0)
jsd_rows = [dict(system=n_, JSD=jsd(ngram_dist(g), real)) for n_, g in [
    ("uroman", [uroman(t) for t in ar_tr]),
    ("T_θ argmax", [T.ar2lat(t, stochastic=False) for t in ar_tr]),
    ("T_θ stochastic", [T.ar2lat(t, rng) for t in ar_tr])]]
display(pd.DataFrame(jsd_rows).round(4))

## 6. Model: encoder + mean pooling + classifier + projection head

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

def load_backbone(name):
    return AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name)

class SBModel(nn.Module):
    def __init__(self, encoder, n_labels, proj_dim):
        super().__init__()
        self.enc = encoder; H = encoder.config.hidden_size
        self.drop = nn.Dropout(0.1); self.cls = nn.Linear(H, n_labels)
        self.proj = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Linear(H, proj_dim))
    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return self.cls(self.drop(pooled)), F.normalize(self.proj(pooled), dim=-1), pooled

## 7. Methods
The factorial core crosses transliteration (argmax or sampled), a symmetric KL term (off or on) and an InfoNCE term (off or on). Every cell also uses cross-entropy on the original text x and on its transliterated view x̃.

| | KL off, InfoNCE off | KL on, InfoNCE off | KL off, InfoNCE on | KL on, InfoNCE on |
|---|---|---|---|---|
| argmax T | `aug_det` | `det_kl` | `augcon_det` | `sb_det` |
| sampled T | `aug_sample` | `sb_noCon` | `sb_noKL` | `scriptbridge` |

The baselines are `vanilla`, the same model with test-time transliteration (`vanilla+TTN`, and `TTN_uroman` when training on Latin script), and `augcon_uroman`, a TransliCo-style baseline that uses uroman romanizations and is defined only when training on Arabic script.

Three comparators complete the design. `rdrop` uses two dropout views of the same text with KL and tests consistency without transliteration. `charnoise` applies random character edits with the ScriptBridge losses and tests a generic perturbation instead of transliteration. `upper_dual` is the supervised dual-script reference, trained on labelled data of both scripts. It is a diagnostic comparator, not a zero-shot system and not a formal upper bound.

The table shows which resources each method may use.

| method | source labels | DoDa, through the transliterator | target-script task text | target labels |
|---|---|---|---|---|
| vanilla, TTN | ✓ | TTN only | – | – |
| transliteration-trained methods | ✓ | ✓ | – | – |
| rdrop, charnoise | ✓ | – | – | – |
| upper_dual | ✓ | – | ✓ | ✓ |

In [ ]:
METHOD_SPEC = {
    "vanilla":       dict(pair=None,     kl=False, con=False),
    "aug_det":       dict(pair="det",    kl=False, con=False),
    "det_kl":        dict(pair="det",    kl=True,  con=False),
    "augcon_det":    dict(pair="det",    kl=False, con=True),
    "sb_det":        dict(pair="det",    kl=True,  con=True),
    "aug_sample":    dict(pair="sample", kl=False, con=False),
    "sb_noCon":      dict(pair="sample", kl=True,  con=False),
    "sb_noKL":       dict(pair="sample", kl=False, con=True),
    "scriptbridge":  dict(pair="sample", kl=True,  con=True),
    "augcon_uroman": dict(pair="uroman", kl=False, con=True),
    "rdrop":         dict(pair="self",   kl=True,  con=False),
    "charnoise":     dict(pair="noise",  kl=True,  con=True),
    "upper_dual":    dict(pair=None,     kl=False, con=False, dual=True),
}
FACTORIAL = {("det", 0, 0): "aug_det", ("det", 1, 0): "det_kl", ("det", 0, 1): "augcon_det", ("det", 1, 1): "sb_det",
             ("sample", 0, 0): "aug_sample", ("sample", 1, 0): "sb_noCon", ("sample", 0, 1): "sb_noKL", ("sample", 1, 1): "scriptbridge"}
def char_noise(t, rng, p=None):
    p = cfg.CHAR_NOISE_P if p is None else p; out = []
    for ch in t:
        r = rng.random()
        if ch.isspace() or r >= p: out.append(ch)
        elif r < p / 4: continue                                   # delete
        elif r < p / 2: out.append(ch + ch)                        # duplicate
        elif r < 3 * p / 4 and out and not out[-1].isspace(): out[-1], ch = ch, out[-1]; out.append(ch)   # swap
        else: out.append(rng.choice("abcdefghijklmnopqrstuvwxyz0123456789") if LA_CHARS.match(ch) or ch.isdigit()
                         else rng.choice("ابتثجحخدذرزسشصضطظعغفقكلمنهوي"))     # substitute within script
    return "".join(out)
def sym_kl(l1, l2):
    p, q = F.log_softmax(l1.float(), -1), F.log_softmax(l2.float(), -1)
    return 0.5 * (F.kl_div(p, q, log_target=True, reduction="batchmean") + F.kl_div(q, p, log_target=True, reduction="batchmean"))
def info_nce(z1, z2, tau):
    sim = z1.float() @ z2.float().T / tau; t = torch.arange(z1.size(0), device=z1.device)
    return 0.5 * (F.cross_entropy(sim, t) + F.cross_entropy(sim.T, t))
def encode(tok, texts):
    b = tok(list(texts), padding=True, truncation=True, max_length=cfg.MAX_LEN, return_tensors="pt")
    return b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)

## 8. Training & evaluation

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, balanced_accuracy_score, average_precision_score
other_script = lambda s: "latin" if s == "arabic" else "arabic"

@torch.no_grad()
def predict(model, tok, texts, bs=128):
    model.eval(); P, E, Z = [], [], []
    for i in range(0, len(texts), bs):
        ids, am = encode(tok, texts[i:i + bs])
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            logits, z, pooled = model(ids, am)
        P.append(logits.float().softmax(-1).cpu()); E.append(pooled.float().cpu()); Z.append(z.float().cpu())
    return torch.cat(P).numpy(), torch.cat(E).numpy(), torch.cat(Z).numpy()

def macro_f1(y, p): return f1_score(y, p.argmax(-1), average="macro")

def make_pair_fn(spec, src, X, rng):
    kind = spec["pair"]
    if kind == "det":
        cache = {x: transliterate(x, src, stochastic=False) for x in X}; return lambda xb: [cache[x] for x in xb]
    if kind == "uroman":
        cache = {x: uroman(x) for x in X}; return lambda xb: [cache[x] for x in xb]
    if kind == "sample": return lambda xb: [transliterate(x, src, rng, True) for x in xb]
    if kind == "self":   return lambda xb: list(xb)
    if kind == "noise":  return lambda xb: [char_noise(x, rng) for x in xb]
    return None

def train_one(method, src, seed, lam_kl=None, lam_con=None):
    spec = METHOD_SPEC[method]
    lam_kl = cfg.LAMBDA_KL if lam_kl is None else lam_kl; lam_con = cfg.LAMBDA_CON if lam_con is None else lam_con
    set_seed(seed); rng = random.Random(seed)
    tok, enc = load_backbone(cfg.MODEL_NAME)
    model = SBModel(enc, len(LABELS), cfg.PROJ_DIM).to(DEVICE)
    tr, dv = SPLITS[src]["train"], SPLITS[src]["dev"]
    if spec.get("dual"):     # supervised dual-script reference (diagnostic): labelled data of both scripts
        tr = pd.concat([tr, SPLITS[other_script(src)]["train"]]); dv = pd.concat([dv, SPLITS[other_script(src)]["dev"]])
    X, Y = tr.text.tolist(), tr.y.tolist()
    pair_fn = make_pair_fn(spec, src, X, rng)
    steps = math.ceil(len(X) / cfg.BATCH_SIZE) * cfg.EPOCHS
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sch = get_linear_schedule_with_warmup(opt, int(cfg.WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and AMP_DTYPE == torch.float16))
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    t_start, n_steps = time.time(), 0
    best, best_state, bad, ep_run = -1, None, 0, 0
    for ep in range(cfg.EPOCHS):
        model.train(); idx = list(range(len(X))); rng.shuffle(idx); t0 = time.time(); ep_run += 1
        for b in range(0, len(idx), cfg.BATCH_SIZE):
            bi = idx[b:b + cfg.BATCH_SIZE]; xb = [X[i] for i in bi]; yb = torch.tensor([Y[i] for i in bi], device=DEVICE)
            texts = xb + pair_fn(xb) if pair_fn else xb
            ids, am = encode(tok, texts)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
                logits, z, _ = model(ids, am)
            B = len(xb); loss = F.cross_entropy(logits[:B].float(), yb); parts = {"ce": loss.item()}
            if pair_fn:
                l = F.cross_entropy(logits[B:].float(), yb); loss = loss + l; parts["ce_t"] = l.item()
                if spec["kl"]: l = sym_kl(logits[:B], logits[B:]); loss = loss + lam_kl * l; parts["kl"] = l.item()
                if spec["con"] and B > 1: l = info_nce(z[:B], z[B:], cfg.TAU); loss = loss + lam_con * l; parts["con"] = l.item()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP); scaler.step(opt); scaler.update(); sch.step(); n_steps += 1
        pdv, _, _ = predict(model, tok, dv.text.tolist()); f = macro_f1(dv.y.values, pdv)   # model selection on SOURCE dev only
        print(f"   [{method}|{src}|{seed}] ep{ep} dev-F1={f:.4f} " + " ".join(f"{k}={v:.3f}" for k, v in parts.items()) + f" ({time.time()-t0:.0f}s)")
        if f > best: best, bad, best_state = f, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= cfg.PATIENCE: break
    model.load_state_dict(best_state)
    info = dict(train_sec=time.time() - t_start, steps=n_steps, epochs_run=ep_run,
                peak_mem_gb=(torch.cuda.max_memory_allocated() / 1e9) if DEVICE == "cuda" else np.nan)
    return model, tok, best, info

# ---------------- metrics ----------------
def auroc(y, P):
    try: return roc_auc_score(y, P[:, 1]) if P.shape[1] == 2 else roc_auc_score(y, P, multi_class="ovr", average="macro")
    except Exception: return np.nan
def prauc(y, P):
    try: return average_precision_score(y, P[:, 1]) if P.shape[1] == 2 else np.mean([average_precision_score(y == k, P[:, k]) for k in range(P.shape[1])])
    except Exception: return np.nan
def brier(y, P): return float(((P - np.eye(P.shape[1])[y]) ** 2).sum(1).mean())
def ece(y, P, bins=10):
    conf, acc = P.max(1), (P.argmax(1) == y); e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
def oracle_f1(y, P):     # DIAGNOSTIC ONLY: threshold tuned on the target test set
    if P.shape[1] != 2: return np.nan
    return max(f1_score(y, (P[:, 1] >= t).astype(int), average="macro") for t in np.unique(np.quantile(P[:, 1], np.linspace(.01, .99, 99))))
def balanced_idx(y, seed=0):
    r = np.random.RandomState(seed); n = min(np.bincount(y)[np.bincount(y) > 0])
    return np.concatenate([r.choice(np.where(y == k)[0], n, replace=False) for k in np.unique(y)])

def cross_metrics(y, P, mixed):
    bi = balanced_idx(y); pure = ~mixed
    return dict(f1_cross=macro_f1(y, P), auc_cross=auroc(y, P), bacc_cross=balanced_accuracy_score(y, P.argmax(1)),
                prauc_cross=prauc(y, P), brier_cross=brier(y, P), ece_cross=ece(y, P),
                f1_cross_bal=macro_f1(y[bi], P[bi]), f1_cross_pure=macro_f1(y[pure], P[pure]) if pure.sum() > 10 else np.nan,
                f1_cross_oracle=oracle_f1(y, P))

DODA_EVAL = doda_test.iloc[:1000]
def _rank_metrics(A, B):
    A = A / np.linalg.norm(A, axis=1, keepdims=True); B = B / np.linalg.norm(B, axis=1, keepdims=True)
    S = A @ B.T; d = np.diag(S)
    r1 = (S > d[:, None]).sum(1) + 1; r2 = (S.T > d[:, None]).sum(1) + 1
    f = lambda r: ((r == 1).mean(), (r <= 5).mean(), (1 / r).mean())
    return [0.5 * (a + b) for a, b in zip(f(r1), f(r2))]
def doda_metrics(model, tok):
    # same content, only the script differs (human-written pairs, no task labels)
    Pa, Ea, Za = predict(model, tok, DODA_EVAL.darija_ar.tolist()); Pl, El, Zl = predict(model, tok, DODA_EVAL.darija.tolist())
    eps = 1e-8; skl = 0.5 * ((Pa * np.log((Pa + eps) / (Pl + eps))).sum(1) + (Pl * np.log((Pl + eps) / (Pa + eps))).sum(1))
    r_raw, r_proj = _rank_metrics(Ea, El), _rank_metrics(Za, Zl)
    return dict(doda_agree=(Pa.argmax(1) == Pl.argmax(1)).mean(), doda_symkl=float(skl.mean()),
                r1_raw=r_raw[0], r5_raw=r_raw[1], mrr_raw=r_raw[2], r1_proj=r_proj[0], r5_proj=r_proj[1], mrr_proj=r_proj[2])

def evaluate(model, tok, src, method, seed, info):
    tgt = other_script(src); te_s, te_t = SPLITS[src]["test"], SPLITS[tgt]["test"]
    P_in, _, _ = predict(model, tok, te_s.text.tolist())
    t0 = time.time(); P_x, _, _ = predict(model, tok, te_t.text.tolist()); lat = 1000 * (time.time() - t0) / len(te_t)
    y, mixed = te_t.y.values, te_t.mixed.values
    base = dict(src=src, seed=seed, f1_in=macro_f1(te_s.y.values, P_in), auc_in=auroc(te_s.y.values, P_in), latency_ms=lat, **info)
    rows = [dict(base, method=method, **cross_metrics(y, P_x, mixed), **doda_metrics(model, tok))]
    preds = {method: P_x}; extra = {}
    if method in ("vanilla", "aug_sample", "scriptbridge"):
        extra[f"{method}+TTN"] = [transliterate(t, tgt, stochastic=False) for t in te_t.text]
    if method == "vanilla" and src == "latin":
        extra["vanilla+TTN_uroman"] = [uroman(t) for t in te_t.text]
    for name, texts in extra.items():
        P, _, _ = predict(model, tok, texts); preds[name] = P
        rows.append(dict(base, method=name, **cross_metrics(y, P, mixed)))
    for r_ in rows: r_["gap"] = r_["f1_in"] - r_["f1_cross"]
    return rows, preds

## 9. Backbone selection (automatic tokenizer-coverage check) and full run
Backbones whose `[UNK]` rate exceeds `UNK_MAX` on either script are skipped. The first `FULL_GRID_MODELS` backbones that pass get the full grid (all methods, 5 seeds); later ones get the reduced grid (`REDUCED_METHODS`, 3 seeds). Cleaned tasks (if any) run the reduced grid, only in the flagged direction, on the full-grid backbones. Runs are resumable: finished runs are skipped.

In [ ]:
def tokenizer_coverage(name, n=2000):
    tok = AutoTokenizer.from_pretrained(name); out = {"model": name}
    _, sp_ = TASK_DATA[next(iter(TASK_DATA))]
    for s in ["arabic", "latin"]:
        texts = sp_[s]["train"].text.tolist()[:n]
        ids = [tok(t, add_special_tokens=False)["input_ids"] for t in texts]
        unk = tok.unk_token_id; tot = sum(len(i) for i in ids)
        out[f"unk_rate_{s}"] = sum(x == unk for i in ids for x in i) / max(1, tot) if unk is not None else 0.0
        out[f"pieces_per_word_{s}"] = tot / max(1, sum(len(t.split()) for t in texts))
    return out

cov_rows, RUN_MODELS = [], []
for name in cfg.MODELS:
    try:
        r_ = tokenizer_coverage(name); cov_rows.append(r_)
        if max(r_["unk_rate_arabic"], r_["unk_rate_latin"]) <= cfg.UNK_MAX and len(RUN_MODELS) < cfg.MAX_MODELS: RUN_MODELS.append(name)
    except Exception as e:
        cov_rows.append({"model": name, "error": f"{type(e).__name__}: {e}"[:120]})
cov = pd.DataFrame(cov_rows); display(cov.round(3))
cov.to_csv(os.path.join(cfg.RESULTS_DIR, "tokenizer_coverage.csv"), index=False)
print("Backbones selected:", RUN_MODELS)

In [ ]:
ALL_RES, ALL_PREDS = {}, {}
SUF = "_noemoji" if cfg.STRIP_EMOJI else ""
base_task = lambda t: CLEAN_TASKS.get(t, t)
for mi, MODEL in enumerate(RUN_MODELS):
    cfg.MODEL_NAME = MODEL; full = mi < cfg.FULL_GRID_MODELS
    for task, (LABELS, SPLITS) in TASK_DATA.items():
        if task in CLEAN_TASKS:
            if not full: continue
            methods, seeds = cfg.REDUCED_METHODS, cfg.REDUCED_SEEDS
            directions = [s_ for (t_, s_) in CONTAMINATED if t_ == CLEAN_TASKS[task]]
        else:
            methods, seeds = (cfg.METHODS, cfg.SEEDS) if full else (cfg.REDUCED_METHODS, cfg.REDUCED_SEEDS)
            directions = cfg.DIRECTIONS
        tag = f"v5_{task}{SUF}_{MODEL.replace('/', '_')}"
        RES_CSV = os.path.join(cfg.RESULTS_DIR, f"results_{tag}.csv"); PRED_PKL = RES_CSV.replace(".csv", "_preds.pkl")
        HP_JSON = RES_CSV.replace(".csv", "_lambdas.json")
        done = pd.read_csv(RES_CSV) if os.path.exists(RES_CSV) else pd.DataFrame()
        preds = pickle.load(open(PRED_PKL, "rb")) if os.path.exists(PRED_PKL) else {}
        lambdas = json.load(open(HP_JSON)) if os.path.exists(HP_JSON) else {}
        for src in directions:
            if cfg.AUTO_TUNE and src not in lambdas:
                scores = []
                for lk, lc in cfg.TUNE_GRID:
                    print(f">>> [{MODEL}|{task}] tuning λ_KL={lk}, λ_con={lc} on {src}-dev")
                    m, _, f, _ = train_one("scriptbridge", src, cfg.SEEDS[0], lk, lc); scores.append((f, lk, lc)); del m; torch.cuda.empty_cache()
                f, lk, lc = max(scores); lambdas[src] = (lk, lc); json.dump(lambdas, open(HP_JSON, "w"))
                print(f"    selected λ_KL={lk}, λ_con={lc} (source dev F1={f:.4f})")
            lk, lc = lambdas.get(src, (cfg.LAMBDA_KL, cfg.LAMBDA_CON))
            for method in methods:
                if method == "augcon_uroman" and src != "arabic": continue
                for seed in seeds:
                    if len(done) and ((done.method == method) & (done.src == src) & (done.seed == seed)).any(): continue
                    print(f">>> [{MODEL}|{task}] {method} | train={src} | seed={seed}")
                    model, tok, _, info = train_one(method, src, seed, lk, lc)
                    rows, p = evaluate(model, tok, src, method, seed, info)
                    for r_ in rows: r_.update(lambda_kl=lk, lambda_con=lc)
                    done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(RES_CSV, index=False)
                    for k, v in p.items(): preds[(k, src, seed)] = v
                    pickle.dump(preds, open(PRED_PKL, "wb"))
                    print(pd.DataFrame(rows)[["method", "f1_in", "f1_cross", "auc_cross", "doda_agree" if "doda_agree" in rows[0] else "gap"]].round(4).to_string(index=False))
                    del model; torch.cuda.empty_cache()
        ALL_RES[(task, MODEL)], ALL_PREDS[(task, MODEL)] = done, preds

## 10. Result tables (mean ± std over seeds) → LaTeX

In [ ]:
ORDER = ["vanilla", "vanilla+TTN", "vanilla+TTN_uroman", "aug_det", "det_kl", "augcon_det", "sb_det",
         "aug_sample", "aug_sample+TTN", "sb_noCon", "sb_noKL", "scriptbridge", "scriptbridge+TTN",
         "augcon_uroman", "rdrop", "charnoise", "upper_dual"]
def fmt(m, s): return "–" if pd.isna(m) else f"{100*m:.2f} ± {100*(0 if pd.isna(s) else s):.2f}"
TABLES = {"main": ["f1_in", "f1_cross", "gap", "auc_cross", "f1_cross_bal", "f1_cross_pure"],
          "invariance": ["doda_agree", "doda_symkl", "r1_raw", "r5_raw", "mrr_raw", "r1_proj", "r5_proj", "mrr_proj"],
          "calibration": ["bacc_cross", "prauc_cross", "brier_cross", "ece_cross", "f1_cross_oracle"]}
for (task, model), res in ALL_RES.items():
    for tname, cols in TABLES.items():
        cols = [c for c in cols if c in res]
        agg = res.groupby(["src", "method"])[cols].agg(["mean", "std"])
        tab = pd.DataFrame({c: [fmt(agg.loc[i, (c, "mean")], agg.loc[i, (c, "std")]) if c != "doda_symkl" else
                                ("–" if pd.isna(agg.loc[i, (c, "mean")]) else f"{agg.loc[i, (c, 'mean')]:.3f}") for i in agg.index] for c in cols}, index=agg.index)
        tab = tab.reindex([(s, m) for s in cfg.DIRECTIONS for m in ORDER if (s, m) in tab.index])
        print(f"===== {tname} | {task} | {model} ====="); display(tab)
        tab.to_latex(os.path.join(cfg.RESULTS_DIR, f"table_{tname}_{task}{SUF}_{model.replace('/', '_')}.tex"), escape=True)
# efficiency (pooled over tasks/directions/seeds)
eff = pd.concat([r.assign(model=m) for (t, m), r in ALL_RES.items()])
eff = eff[eff.train_sec.notna()].groupby(["model", "method"])[["train_sec", "steps", "epochs_run", "peak_mem_gb", "latency_ms"]].mean().round(2)
print("===== efficiency (means over runs) ====="); display(eff)
eff.to_latex(os.path.join(cfg.RESULTS_DIR, "table_efficiency.tex"))

## 11. Statistics: pre-declared contrasts, hierarchical bootstrap, Holm correction
Six primary contrasts were declared before the final experiments. C1 compares training-time with test-time transliteration (`aug_sample` − `vanilla+TTN`). C2 compares the learned with the universal transliterator (`augcon_det` − `augcon_uroman`). C3 compares sampling with argmax under cross-entropy only (`aug_sample` − `aug_det`). C4 makes the same comparison under the full objective (`scriptbridge` − `sb_det`). C5 measures the added value of KL and InfoNCE (`scriptbridge` − `aug_sample`). C6 compares transliteration with a generic perturbation (`scriptbridge` − `charnoise`).

An effect is the mean over seeds of the per-seed difference, so it describes a typical independently trained model rather than a seed ensemble. The 95% interval and two-sided p-value come from a hierarchical bootstrap that resamples seeds and test examples. Holm correction is applied within each family, defined as all primary contrasts of one backbone and one metric over the clean settings. Contaminated directions and cleaned tasks are reported separately and are not part of the corrected families. A cluster bootstrap by video or post is not possible because the datasets do not release collection identifiers.

In [ ]:
from scipy.stats import rankdata, t as _t, spearmanr
def fast_f1(y, pred, K):
    f = []
    for k in range(K):
        tp = np.sum((pred == k) & (y == k)); fp = np.sum((pred == k) & (y != k)); fn = np.sum((pred != k) & (y == k))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))
def fast_auc(y, P):
    def _bin(yb, s):
        pos = yb == 1; n1 = pos.sum(); n0 = len(yb) - n1
        if n1 == 0 or n0 == 0: return np.nan
        r = rankdata(s); return (r[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    return _bin(y, P[:, 1]) if P.shape[1] == 2 else np.nanmean([_bin((y == k).astype(int), P[:, k]) for k in range(P.shape[1])])
METRIC_FNS = {"F1": lambda y, P: fast_f1(y, P.argmax(1), P.shape[1]), "AUROC": fast_auc}
def seed_preds(preds, m, src): return {s: v for (k, ss, s), v in preds.items() if k == m and ss == src}
def hier_boot(y, A, B, fn, n_boot, seed=0):
    seeds = sorted(set(A) & set(B))
    if not seeds: return (np.nan,) * 4 + (0,)
    point = np.mean([fn(y, A[s]) - fn(y, B[s]) for s in seeds])
    r = np.random.RandomState(seed); N = len(y); d = np.empty(n_boot)
    for b in range(n_boot):
        ss = r.choice(seeds, len(seeds)); idx = r.randint(0, N, N); yb = y[idx]
        d[b] = np.mean([fn(yb, A[s][idx]) - fn(yb, B[s][idx]) for s in ss])
    lo, hi = np.nanpercentile(d, [2.5, 97.5]); p = min(1.0, 2 * min(np.mean(d <= 0), np.mean(d >= 0)))
    return point, lo, hi, p, len(seeds)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, j in enumerate(o): run = max(run, (m - i) * p[j]); adj[j] = min(1.0, run)
    return adj
PRIMARY = [("C1", "aug_sample", "vanilla+TTN"), ("C2", "augcon_det", "augcon_uroman"), ("C3", "aug_sample", "aug_det"),
           ("C4", "scriptbridge", "sb_det"), ("C5", "scriptbridge", "aug_sample"), ("C6", "scriptbridge", "charnoise")]
SECONDARY = [("S1", "scriptbridge", "vanilla"), ("S2", "scriptbridge", "vanilla+TTN"), ("S3", "scriptbridge", "augcon_det"),
             ("S4", "aug_sample", "vanilla"), ("S5", "scriptbridge", "rdrop"), ("S6", "upper_dual", "scriptbridge")]
sig_rows = []
for (task, model), preds in ALL_PREDS.items():
    LABELS, SPLITS = TASK_DATA[task]
    for src in cfg.DIRECTIONS:
        y = SPLITS[other_script(src)]["test"].y.values
        clean = (base_task(task), src) not in CONTAMINATED and task not in CLEAN_TASKS
        for fam, lst in [("primary", PRIMARY), ("secondary", SECONDARY)]:
            for cid, a, b in lst:
                A, B = seed_preds(preds, a, src), seed_preds(preds, b, src)
                if not A or not B: continue
                for mn, fn in METRIC_FNS.items():
                    pt, lo, hi, p, ns = hier_boot(y, A, B, fn, cfg.N_BOOT)
                    sig_rows.append(dict(task=task, model=model, direction=f"{src}→{other_script(src)}", clean=clean,
                                         family=fam if clean else ("cleaned" if task in CLEAN_TASKS else "contaminated"),
                                         contrast=cid, a=a, b=b, metric=mn, delta=100 * pt, ci_low=100 * lo, ci_high=100 * hi, p=p, n_seeds=ns))
sig = pd.DataFrame(sig_rows)
if len(sig):
    sig["p_holm"] = np.nan
    for _, g in sig[sig.family == "primary"].groupby(["model", "metric"]):
        sig.loc[g.index, "p_holm"] = holm(g.p.values)
    sig["significant"] = np.where(sig.family == "primary", sig.p_holm < 0.05, sig.p < 0.05)
with pd.option_context("display.max_rows", 500, "display.width", 250): display(sig.round(4))
sig.to_csv(os.path.join(cfg.RESULTS_DIR, "significance_v5.csv"), index=False)

### 11.1 Factorial main effects (sampling, KL, InfoNCE) and retrieval–classification association

In [ ]:
fx_rows, corr_rows = [], []
for (task, model), res in ALL_RES.items():
    for src in cfg.DIRECTIONS:
        r = res[res.src == src]
        if not all(m in set(r.method) for m in FACTORIAL.values()): continue
        seeds = sorted(set.intersection(*[set(r[r.method == m].seed) for m in FACTORIAL.values()]))
        for metric in ["f1_cross", "auc_cross", "f1_cross_bal", "doda_agree", "r1_raw"]:
            v = {k: r[r.method == m].set_index("seed")[metric] for k, m in FACTORIAL.items()}
            eff = {"sampling": lambda s: np.mean([v[("sample", a, c)][s] - v[("det", a, c)][s] for a in (0, 1) for c in (0, 1)]),
                   "KL":       lambda s: np.mean([v[(t_, 1, c)][s] - v[(t_, 0, c)][s] for t_ in ("det", "sample") for c in (0, 1)]),
                   "InfoNCE":  lambda s: np.mean([v[(t_, a, 1)][s] - v[(t_, a, 0)][s] for t_ in ("det", "sample") for a in (0, 1)]),
                   "sampling×InfoNCE": lambda s: np.mean([(v[("sample", a, 1)][s] - v[("sample", a, 0)][s]) - (v[("det", a, 1)][s] - v[("det", a, 0)][s]) for a in (0, 1)])}
            for name, fn in eff.items():
                x = np.array([fn(s) for s in seeds]); n = len(x)
                half = _t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
                fx_rows.append(dict(task=task, model=model, direction=f"{src}→{other_script(src)}",
                                    clean=(base_task(task), src) not in CONTAMINATED and task not in CLEAN_TASKS,
                                    metric=metric, effect=name, mean=100 * x.mean(), ci_low=100 * (x.mean() - half), ci_high=100 * (x.mean() + half), n_seeds=n))
        rr = r.dropna(subset=["r1_raw", "f1_cross"])
        if len(rr) > 5:
            rho, p = spearmanr(rr.r1_raw, rr.f1_cross); corr_rows.append(dict(task=task, model=model, direction=src, spearman_r1_vs_f1=rho, p=p, n_runs=len(rr)))
fx = pd.DataFrame(fx_rows).round(2)
with pd.option_context("display.max_rows", 500, "display.width", 250): display(fx)
fx.to_csv(os.path.join(cfg.RESULTS_DIR, "factorial_effects.csv"), index=False)
display(pd.DataFrame(corr_rows).round(3)); pd.DataFrame(corr_rows).to_csv(os.path.join(cfg.RESULTS_DIR, "retrieval_vs_f1.csv"), index=False)

## 12. Automatic error slices on the cross-script test set (seed-ensembled predictions, descriptive only)

In [ ]:
def ens(preds, m, src):
    ps = [v for (k, s, _), v in preds.items() if k == m and s == src]; return np.mean(ps, 0) if ps else None
FR = set("le la les de des du et est pas je tu il elle une un pour avec mais que qui merci tres très bien c'est cest vous nous".split())
SLICES = {
    "digit-letters (3/7/9…)": lambda t: bool(re.search(r"[a-z][2-9]|[2-9][a-z]", t.lower())),
    "French tokens":          lambda t: len(FR & set(re.findall(r"[a-zéèàç']+", t.lower()))) > 0,
    "elongation":             lambda t: bool(re.search(r"(\w)\1{2,}", t)),
    "mixed script":           is_mixed,
    "short (≤5 tokens)":      lambda t: len(t.split()) <= 5,
    "long (>15 tokens)":      lambda t: len(t.split()) > 15,
}
for (task, model), preds in ALL_PREDS.items():
    LABELS, SPLITS = TASK_DATA[task]; out = []
    for src in cfg.DIRECTIONS:
        te = SPLITS[other_script(src)]["test"]
        for name, fn in SLICES.items():
            mask = te.text.map(fn).values
            if mask.sum() < 30: continue
            row = dict(task=task, model=model, direction=f"{src}→{other_script(src)}", slice=name, n=int(mask.sum()))
            for m in ["vanilla", "vanilla+TTN", "aug_det", "aug_sample", "sb_det", "scriptbridge"]:
                p = ens(preds, m, src)
                if p is not None: row[m] = round(100 * f1_score(te.y.values[mask], p[mask].argmax(-1), average="macro"), 2)
            out.append(row)
    if out:
        display(pd.DataFrame(out)); pd.DataFrame(out).to_csv(os.path.join(cfg.RESULTS_DIR, f"error_slices_{task}_{model.replace('/', '_')}.csv"), index=False)

## 13. Figures

In [ ]:
import matplotlib.pyplot as plt
FIG_M = ["vanilla", "vanilla+TTN", "aug_det", "aug_sample", "sb_det", "scriptbridge", "upper_dual"]
for (task, model), res in ALL_RES.items():
    fig, axes = plt.subplots(1, len(cfg.DIRECTIONS), figsize=(7 * len(cfg.DIRECTIONS), 4.2), squeeze=False)
    for ax, src in zip(axes[0], cfg.DIRECTIONS):
        sub = res[res.src == src].groupby("method")[["f1_in", "f1_cross"]].agg(["mean", "std"])
        ms = [m for m in FIG_M if m in sub.index]
        if not ms: ax.axis("off"); continue
        x = np.arange(len(ms)); w = .38
        ax.bar(x - w/2, 100*sub.loc[ms, ("f1_in", "mean")], w, yerr=100*sub.loc[ms, ("f1_in", "std")].fillna(0), label="in-script", color="#9aa5b1")
        ax.bar(x + w/2, 100*sub.loc[ms, ("f1_cross", "mean")], w, yerr=100*sub.loc[ms, ("f1_cross", "std")].fillna(0), label="cross-script", color="#2f6fb0")
        ax.set_xticks(x); ax.set_xticklabels(ms, rotation=40, ha="right"); ax.set_ylabel("Macro-F1 (%)")
        ax.set_title(f"{task} | {model.split('/')[-1]}: train {src} → test {other_script(src)}", fontsize=10)
        ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(os.path.join(cfg.RESULTS_DIR, f"fig_f1_{task}_{model.replace('/', '_')}.pdf")); plt.show()
print("All outputs in", cfg.RESULTS_DIR); print(sorted(os.listdir(cfg.RESULTS_DIR)))

## 14. Reproducibility notes
All splits are fixed (seed 1234), so every system sees identical data. Per-seed predicted probabilities are saved in `*_preds.pkl`, and all metrics per run are saved in `results_v5_*.csv` together with the selected λ values, training time, steps, epochs and peak GPU memory. The learned transliterator is saved to `transliterator_params.json`, the split manifests (text hashes and labels) to `split_manifest_*.csv`, and the library versions to `environment.json`.

In [ ]:
import transformers, sklearn, platform as _pf
versions = {"python": _pf.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
            "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print(versions); json.dump(versions, open(os.path.join(cfg.RESULTS_DIR, "environment.json"), "w"), indent=1)
for task, (L_, SP_) in TASK_DATA.items():   # split manifest (text hashes) for exact reproduction
    import hashlib
    man = [dict(task=task, script=s_, split=k_, sha1=hashlib.sha1(t.encode()).hexdigest()[:16], y=int(y_))
           for s_ in SP_ for k_ in SP_[s_] for t, y_ in zip(SP_[s_][k_].text, SP_[s_][k_].y)]
    pd.DataFrame(man).to_csv(os.path.join(cfg.RESULTS_DIR, f"split_manifest_{task}.csv"), index=False)